In [ ]:
import json

ticker = dbutils.widgets.get("ticker")
timeout = int(dbutils.widgets.get("timeout"))
catalog = dbutils.widgets.get("catalog")

In [ ]:
def compute_statistics(df):
    first_price = df["Close"].iloc[0]
    last_price = df["Close"].iloc[-1]

    sdf = spark.createDataFrame(
        [(
            ticker,
            df.index.min(),
            df.index.max(),
            float(df["Close"].max()),
            float(df["Close"].min()),
            float(last_price),
            float((last_price - first_price) / first_price * 100),
            float((df["Close"] * df["Volume"]).sum() / df["Volume"].sum()),
            float(df["Volume"].sum()),
        )],
        [
            "ticker", "window_start", "window_end", "high",
            "low", "last_price", "pct_change", "vwap", "volume"
        ]
    )

    from delta.tables import DeltaTable
    DeltaTable.forName(f"{catalog}.yfinance.nrt_statistics").alias("target")\
        .merge(
            sdf.alias("source"),
            "source.ticker = target.ticker"
        ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

In [ ]:
from datetime import datetime, timedelta
import yfinance as yf

_now = datetime.now()
try:
    while (datetime.now() - _now).total_seconds() <= timeout:
        end = datetime.now()
        start = end - timedelta(hours=1)
        df = yf.download(ticker, start=start, end=end, interval="1m", auto_adjust=False)
        compute_statistics(df)
    dbutils.notebook.exit(json.dumps({"status": 1}))
except Exception as e:
    print(f"{ticker} nrt loop failed: {e}")
    dbutils.notebook.exit(json.dumps({"status": 0}))